# Credit Card Fraud Detection — Exploratory Data Analysis

This notebook explores the creditcard.csv dataset before model training. Key goals:
- Understand the class imbalance
- Identify the most discriminative features
- Motivate preprocessing choices (scaling, imbalance strategy)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

DATA_PATH = "../DATA/creditcard.csv"
FIG_DIR = "../outputs/figures"
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Load Data & Basic Profile

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"\nFraud count : {df['Class'].sum():>6}")
print(f"Legit count : {(df['Class']==0).sum():>6}")
print(f"Fraud rate  : {df['Class'].mean()*100:.3f}%")
df.head()

In [ ]:
print("Data types and null counts:")
print(df.dtypes.to_string())
print(f"\nNull values: {df.isnull().sum().sum()}")
df.describe()

## 2. Class Distribution

In [ ]:
counts = df["Class"].value_counts().sort_index()
labels = ["Legitimate", "Fraud"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
axes[0].bar(labels, counts.values, color=["steelblue", "tomato"])
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 1000, f"{v:,}", ha="center", fontsize=11)
axes[0].set_title("Transaction Counts")
axes[0].set_ylabel("Count")

# Pie chart
axes[1].pie(
    counts.values, labels=labels, autopct="%1.3f%%",
    colors=["steelblue", "tomato"], startangle=90
)
axes[1].set_title("Class Distribution")

fig.suptitle("Severe Class Imbalance: 0.172% Fraud Rate", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/class_distribution.png", dpi=150)
plt.show()

**Finding:** The dataset is severely imbalanced — only 492 frauds out of 284,807 transactions (0.172%). This makes accuracy a misleading metric and motivates AUPRC as our primary KPI. A dummy classifier predicting "all legit" would score 99.83% accuracy but catch zero frauds.

## 3. Time and Amount Distributions

In [ ]:
fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Time distributions
axes[0, 0].hist(legit["Time"] / 3600, bins=48, alpha=0.6, color="steelblue", label="Legitimate")
axes[0, 0].hist(fraud["Time"] / 3600, bins=48, alpha=0.8, color="tomato", label="Fraud")
axes[0, 0].set_xlabel("Time (hours from first transaction)")
axes[0, 0].set_ylabel("Count")
axes[0, 0].set_title("Time Distribution")
axes[0, 0].legend()

# Time KDE
sns.kdeplot(legit["Time"] / 3600, ax=axes[0, 1], color="steelblue", label="Legitimate", fill=True, alpha=0.3)
sns.kdeplot(fraud["Time"] / 3600, ax=axes[0, 1], color="tomato", label="Fraud", fill=True, alpha=0.5)
axes[0, 1].set_xlabel("Time (hours)")
axes[0, 1].set_title("Time Distribution (KDE)")
axes[0, 1].legend()

# Amount histogram (log scale)
axes[1, 0].hist(legit["Amount"].clip(upper=500), bins=50, alpha=0.6, color="steelblue", label="Legitimate")
axes[1, 0].hist(fraud["Amount"].clip(upper=500), bins=50, alpha=0.8, color="tomato", label="Fraud")
axes[1, 0].set_xlabel("Amount (clipped at $500)")
axes[1, 0].set_ylabel("Count")
axes[1, 0].set_title("Transaction Amount Distribution")
axes[1, 0].legend()

# Amount stats
axes[1, 1].boxplot(
    [legit["Amount"].clip(upper=500), fraud["Amount"].clip(upper=500)],
    labels=["Legitimate", "Fraud"],
    patch_artist=True,
    boxprops=dict(facecolor="steelblue", alpha=0.6)
)
axes[1, 1].set_ylabel("Amount (clipped at $500)")
axes[1, 1].set_title("Amount Boxplot")

fig.suptitle("Time and Amount: Fraud vs Legitimate", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/amount_time_dist.png", dpi=150)
plt.show()

print(f"Legit median amount : ${legit['Amount'].median():.2f}")
print(f"Fraud median amount : ${fraud['Amount'].median():.2f}")

**Finding:** Fraud transactions tend to be smaller amounts — consistent with small test transactions before large fraud. Both Time and Amount need StandardScaler before feeding to linear models and distance-based algorithms.

## 4. Correlation Matrix

In [ ]:
corr = df.corr()

# Show which features correlate most with Class
class_corr = corr["Class"].drop("Class").sort_values(key=abs, ascending=False)
print("Top 10 features by |correlation| with Class:")
print(class_corr.head(10).to_string())

mask = np.triu(np.ones_like(corr, dtype=bool))
fig, ax = plt.subplots(figsize=(18, 14))
sns.heatmap(
    corr, mask=mask, cmap="RdBu_r", center=0,
    vmin=-1, vmax=1, linewidths=0.4, ax=ax,
    annot=False
)
ax.set_title("Feature Correlation Matrix", fontsize=14)
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/correlation_matrix.png", dpi=150)
plt.show()

**Finding:** V17, V14, V12, and V10 are the most strongly negatively correlated with Class (fraud=1). These PCA components are the most discriminative features. V1–V28 are largely uncorrelated with each other (PCA guarantee), which benefits most classifiers.

## 5. PCA Feature Distributions (V1–V28)

In [ ]:
v_features = [f"V{i}" for i in range(1, 29)]

fig, axes = plt.subplots(7, 4, figsize=(20, 24))
for i, col in enumerate(v_features):
    ax = axes.flat[i]
    corr_val = df[col].corr(df["Class"])
    sns.kdeplot(legit[col], ax=ax, color="steelblue", label="Legit", fill=True, alpha=0.3)
    sns.kdeplot(fraud[col], ax=ax, color="tomato", label="Fraud", fill=True, alpha=0.5)
    ax.set_title(f"{col}  (r={corr_val:.3f})", fontsize=9)
    ax.set_xlabel("")
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle("PCA Component Distributions: Fraud vs Legitimate", fontsize=14, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/feature_distributions.png", dpi=120)
plt.show()

**Finding:** Several PCA components (V4, V11, V14, V17) show clearly separated distributions between fraud and legitimate transactions. Others overlap substantially. Features with high |r| are most valuable for classification.

## 6. Outlier Analysis — Top Discriminative Features

In [ ]:
top_features = class_corr.head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for i, col in enumerate(top_features):
    ax = axes.flat[i]
    data_to_plot = [legit[col], fraud[col]]
    bp = ax.boxplot(data_to_plot, labels=["Legit", "Fraud"], patch_artist=True)
    bp["boxes"][0].set_facecolor("steelblue")
    bp["boxes"][0].set_alpha(0.6)
    bp["boxes"][1].set_facecolor("tomato")
    bp["boxes"][1].set_alpha(0.6)
    ax.set_title(f"{col}", fontsize=10)

fig.suptitle("Boxplots: Top Discriminative Features", fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(f"{FIG_DIR}/outlier_analysis.png", dpi=150)
plt.show()

**Finding:** Fraud transactions show dramatically different medians and ranges for V14, V17, V12, and V10. These extreme differences mean even simple linear separators in these dimensions should achieve high discrimination.

## 7. Summary of EDA Findings

| Finding | Implication for Modeling |
|---|---|
| 0.172% fraud rate (severe imbalance) | Use AUPRC as primary metric; use `class_weight='balanced'` or `scale_pos_weight` |
| No missing values | No imputation needed |
| V1–V28 already PCA-normalized, zero-centered | Scale only `Time` and `Amount` with StandardScaler |
| V17, V14, V12, V10 most correlated with fraud | Tree-based models will naturally exploit these |
| Fraud amounts tend to be smaller | `Amount` feature is informative |
| V1–V28 are largely uncorrelated with each other | No feature collinearity issues for logistic regression |
| Baseline AUPRC ≈ class prevalence ≈ 0.0017 | Any useful model should exceed 0.7 AUPRC |

**Next steps:** Run `python DATA/preprocess.py` to generate processed splits, then `python MODELS/train_models.py` to train and compare models.